# Battery deployment 2026 vs 2025

In [1]:
import pandas as pd
import pathlib
import mlflow
import sklearn.neighbors
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [2]:
def read_data(rel_path):
    path = pathlib.Path("../data").resolve() / (rel_path + ".parquet")
    df = pd.read_parquet(path)
    return df

In [3]:
data_sessions_2026 = read_data("gold/session_metadata_Y2026")
data_sessions_2025 = read_data("gold/session_metadata_Y2025")


In [4]:
data_sessions_2026[['round_number', 'event_location']].drop_duplicates()

,round_number,event_location
0,1,Melbourne
5,2,Shanghai
10,3,Suzuka
15,4,Miami Gardens
20,0,Bahrain


In [5]:
data_sessions_2025[['round_number', 'event_location']].drop_duplicates()

,round_number,event_location
0,1,Melbourne
5,2,Shanghai
10,3,Suzuka
15,4,Sakhir
20,5,Jeddah
25,6,Miami Gardens
30,7,Imola
35,8,Monaco
40,9,Barcelona
45,10,Montréal


In [6]:
data_laps_2026 = read_data("gold/session_laps_Y2026")
data_laps_2025 = read_data("gold/session_laps_Y2025")


In [7]:
data_telemetry_pos_2026 = read_data("gold/telemetry_pos_Y2026R04")
data_telemetry_pos_2025 = read_data("gold/telemetry_pos_Y2025R06")
data_telemetry_car_2026 = read_data("gold/telemetry_car_Y2026R04")
data_telemetry_car_2025 = read_data("gold/telemetry_car_Y2025R06")


In [8]:
data_telemetry_pos_2025

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status
0,2025,Y2025R06S1,1,1,2025-05-02 16:31:13.558000+00:00,928.507,0.013,1,3791.0,-1225.0,0.0,OnTrack
1,2025,Y2025R06S1,1,1,2025-05-02 16:31:13.959000+00:00,928.908,0.414,1,3796.0,-1228.0,0.0,OnTrack
2,2025,Y2025R06S1,1,1,2025-05-02 16:31:14.179000+00:00,929.128,0.634,1,3801.0,-1230.0,0.0,OnTrack
3,2025,Y2025R06S1,1,1,2025-05-02 16:31:14.519000+00:00,929.468,0.974,1,3804.0,-1233.0,0.0,OnTrack
4,2025,Y2025R06S1,1,1,2025-05-02 16:31:14.779000+00:00,929.728,1.234,1,3808.0,-1235.0,0.0,OnTrack
...,...,...,...,...,...,...,...,...,...,...,...,...
1121683,2025,Y2025R06S5,87,28,2025-05-04 20:49:22.138000+00:00,6175.297,148.936,6,-3965.0,-3880.0,242.0,OnTrack
1121684,2025,Y2025R06S5,87,28,2025-05-04 20:49:22.478000+00:00,6175.637,149.276,6,-3965.0,-3880.0,242.0,OnTrack
1121685,2025,Y2025R06S5,87,28,2025-05-04 20:49:22.578000+00:00,6175.737,149.376,6,-3965.0,-3880.0,242.0,OnTrack
1121686,2025,Y2025R06S5,87,28,2025-05-04 20:49:22.738000+00:00,6175.897,149.536,6,-3965.0,-3880.0,242.0,OnTrack


In [9]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(experiment_names=["formula_one_circuit_map"])
runs = runs[runs['tags.session_ids'].str.contains('Y2026R04S1_Y2026R04S2_Y2026R04S3_Y2026R04S4_Y2026R04S5')]
print(runs.iloc[0].to_dict())
df_circuit_map_2026  =  pd.read_parquet(
    mlflow.artifacts.download_artifacts(artifact_uri=runs.iloc[0]["artifact_uri"] + "/result.parquet")
)

{'run_id': '5579cfcacc814dd9b380cfa7e9824346', 'experiment_id': '1', 'status': 'FINISHED', 'artifact_uri': '/Users/tiagobbatalhao/Documents/projects/formula_one_data_analysis/mlruns/1/5579cfcacc814dd9b380cfa7e9824346/artifacts', 'start_time': Timestamp('2026-05-10 02:06:36.558000+0000', tz='UTC'), 'end_time': Timestamp('2026-05-10 02:07:53.682000+0000', tz='UTC'), 'metrics.mae-time-x': 133.53159839768182, 'metrics.mae-time-y': 57.36564540077072, 'metrics.mae-time-z': 0.5573925391098835, 'metrics.rmse-time-x': 513.0522153507158, 'metrics.rmse-distance-x': 4.820895405566607, 'metrics.mae-distance-z': 0.06902765621663212, 'metrics.rmse-time-z': 1.0704411731234216, 'metrics.mae-distance-y': 1.7094279623683164, 'metrics.rmse-distance-z': 0.13183740450247466, 'metrics.rmse-time-y': 259.6838476163175, 'metrics.adjustment': 7.8581148040736e-05, 'metrics.rmse-distance-y': 2.753459480761823, 'metrics.mae-distance-x': 2.5433057842099243, 'params.max_degree': '100', 'params.predict_size': '100000'

In [10]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(experiment_names=["formula_one_circuit_map"])
runs = runs[runs['tags.session_ids'].str.contains('Y2025R06S1_Y2025R06S2_Y2025R06S3_Y2025R06S4_Y2025R06S5')]
print(runs.iloc[0].to_dict())
df_circuit_map_2025  =  pd.read_parquet(
    mlflow.artifacts.download_artifacts(artifact_uri=runs.iloc[0]["artifact_uri"] + "/result.parquet")
)

{'run_id': '38cdaf2a2fe648c0852378b1d77a3276', 'experiment_id': '1', 'status': 'FINISHED', 'artifact_uri': '/Users/tiagobbatalhao/Documents/projects/formula_one_data_analysis/mlruns/1/38cdaf2a2fe648c0852378b1d77a3276/artifacts', 'start_time': Timestamp('2026-05-10 02:03:32.419000+0000', tz='UTC'), 'end_time': Timestamp('2026-05-10 02:04:49.676000+0000', tz='UTC'), 'metrics.mae-time-x': 286.5386696362372, 'metrics.mae-time-y': 123.13734599792575, 'metrics.mae-time-z': 14.116480384951602, 'metrics.rmse-time-x': 982.7950094597411, 'metrics.rmse-distance-x': 7.352295385511819, 'metrics.mae-distance-z': 0.1913092280509598, 'metrics.rmse-time-z': 41.79190854974199, 'metrics.mae-distance-y': 2.1312404794974706, 'metrics.rmse-distance-z': 0.31974756569550344, 'metrics.rmse-time-y': 435.18375512452275, 'metrics.adjustment': 0.0005740490172844165, 'metrics.rmse-distance-y': 3.8415977875140532, 'metrics.mae-distance-x': 4.2852620827114265, 'params.max_degree': '100', 'params.predict_size': '10000

In [11]:
def run_circuit_encoding(telemetry_pos: pd.DataFrame, circuit_map: pd.DataFrame) -> pd.DataFrame:
    columns_coordinates = ["coordinate_x", "coordinate_y"]
    columns_pkey = ["session_id", "driver_number", "lap_number", "timestamp"]
    find_neighbours = (
        sklearn.neighbors.NearestNeighbors(n_neighbors=1)
        .fit(circuit_map[columns_coordinates].values)
        .kneighbors(telemetry_pos[columns_coordinates].values)
    )
    df_pos = telemetry_pos[columns_pkey].copy()
    df_pos["idx"] = find_neighbours[1][:, 0]
    df_pos = df_pos.merge(
        circuit_map.assign(idx=lambda df: range(len(df)))[["idx", "encoding", "distance_m"]]
    )
    return df_pos


In [12]:
if "distance_m" not in data_telemetry_pos_2026.columns:
    data_telemetry_pos_2026 = data_telemetry_pos_2026.merge(
        run_circuit_encoding(data_telemetry_pos_2026, df_circuit_map_2026)
    )
if "distance_m" not in data_telemetry_pos_2025.columns:
    data_telemetry_pos_2025 = data_telemetry_pos_2025.merge(
        run_circuit_encoding(data_telemetry_pos_2025, df_circuit_map_2025)
    )


In [13]:
lap_2026 = data_laps_2026[
    (data_laps_2026['session_id']=='Y2026R04S4')
].sort_values(by=['time_lap'], ascending=True).iloc[:1]
lap_2026.T

,9631
year,2026
session_id,Y2026R04S4
driver_number,12
driver_name,ANT
driver_team,Mercedes
lap_number,14
stint,5.0
timestamp_lap_start,2026-05-02 20:55:34.551000+00:00
timing_start_lap,4096.754
timing_end_lap,4184.552


In [14]:
lap_2025 = data_laps_2025[
    (data_laps_2025['session_id']=='Y2025R06S4')
].sort_values(by=['time_lap'], ascending=True).iloc[:1]
lap_2025.T

,13442
year,2025
session_id,Y2025R06S4
driver_number,1
driver_name,VER
driver_team,Red Bull Racing
lap_number,17
stint,6.0
timestamp_lap_start,2025-05-03 21:13:09.721000+00:00
timing_start_lap,4309.694
timing_end_lap,4395.898


In [15]:
data_telemetry_pos_lap2026 = (
    data_telemetry_pos_2026.merge(
        lap_2026[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)
data_telemetry_car_lap2026 = (
    data_telemetry_car_2026.merge(
        lap_2026[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)

In [16]:
data_telemetry_pos_lap2025 = (
    data_telemetry_pos_2025.merge(
        lap_2025[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)
data_telemetry_car_lap2025 = (
    data_telemetry_car_2025.merge(
        lap_2025[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)

In [17]:
data_telemetry_pos_lap2026

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status,idx,encoding,distance_m
0,2026,Y2026R04S4,12,14,2026-05-02 20:55:34.634000+00:00,4096.837,0.083,1,1996.0,28.0,250.0,OnTrack,287,0.00287,15.165395
1,2026,Y2026R04S4,12,14,2026-05-02 20:55:35.014000+00:00,4097.217,0.463,1,2157.0,-73.0,251.0,OnTrack,645,0.00645,34.156824
2,2026,Y2026R04S4,12,14,2026-05-02 20:55:35.294000+00:00,4097.497,0.743,1,2347.0,-192.0,251.0,OnTrack,1068,0.01068,56.599719
3,2026,Y2026R04S4,12,14,2026-05-02 20:55:35.354000+00:00,4097.557,0.803,1,2388.0,-218.0,251.0,OnTrack,1160,0.01160,61.455970
4,2026,Y2026R04S4,12,14,2026-05-02 20:55:35.614000+00:00,4097.817,1.063,1,2567.0,-331.0,251.0,OnTrack,1560,0.01560,82.633302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,2026,Y2026R04S4,12,14,2026-05-02 20:57:01.314000+00:00,4183.517,86.763,1,1194.0,460.0,246.0,OnTrack,98565,0.98565,5207.833238
320,2026,Y2026R04S4,12,14,2026-05-02 20:57:01.614000+00:00,4183.817,87.063,1,1400.0,375.0,248.0,OnTrack,98985,0.98985,5230.126205
321,2026,Y2026R04S4,12,14,2026-05-02 20:57:01.674000+00:00,4183.877,87.123,1,1418.0,367.0,248.0,OnTrack,99022,0.99022,5232.084883
322,2026,Y2026R04S4,12,14,2026-05-02 20:57:02.014000+00:00,4184.217,87.463,1,1636.0,251.0,249.0,OnTrack,99488,0.99488,5256.742270


In [18]:
data_telemetry_pos_lap2025

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status,idx,encoding,distance_m
0,2025,Y2025R06S4,1,17,2025-05-03 21:13:09.756000+00:00,4309.729,0.035,1,1895.0,92.0,250.0,OnTrack,31,0.00031,1.592116
1,2025,Y2025R06S4,1,17,2025-05-03 21:13:10.037000+00:00,4310.010,0.316,1,2118.0,-48.0,251.0,OnTrack,533,0.00533,28.066827
2,2025,Y2025R06S4,1,17,2025-05-03 21:13:10.457000+00:00,4310.430,0.736,1,2457.0,-262.0,251.0,OnTrack,1304,0.01304,68.143945
3,2025,Y2025R06S4,1,17,2025-05-03 21:13:10.737000+00:00,4310.710,1.016,1,2586.0,-343.0,251.0,OnTrack,1588,0.01588,83.316037
4,2025,Y2025R06S4,1,17,2025-05-03 21:13:10.977000+00:00,4310.950,1.256,1,2759.0,-451.0,250.0,OnTrack,1982,0.01982,103.688816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,2025,Y2025R06S4,1,17,2025-05-03 21:14:34.776000+00:00,4394.749,85.055,1,1066.0,504.0,246.0,OnTrack,98270,0.98270,5107.073664
321,2025,Y2025R06S4,1,17,2025-05-03 21:14:34.976000+00:00,4394.949,85.255,1,1210.0,454.0,247.0,OnTrack,98554,0.98554,5122.187902
322,2025,Y2025R06S4,1,17,2025-05-03 21:14:35.256000+00:00,4395.229,85.535,1,1408.0,371.0,248.0,OnTrack,98964,0.98964,5143.492628
323,2025,Y2025R06S4,1,17,2025-05-03 21:14:35.517000+00:00,4395.490,85.796,1,1629.0,255.0,249.0,OnTrack,99438,0.99438,5168.330133


In [19]:
_start_2026, _end_2026 = 3396, 4785
df1 = data_telemetry_pos_lap2026[
    (data_telemetry_pos_lap2026['distance_m'] > _start_2026 - 50)
    & (data_telemetry_pos_lap2026['distance_m'] < _end_2026 + 50)
]
df2 = data_telemetry_car_lap2026[
    (data_telemetry_car_lap2026['timing_from_lap'] > df1['timing_from_lap'].min())
    & (data_telemetry_car_lap2026['timing_from_lap'] < df1['timing_from_lap'].max())
]
df2['distance_m'] = np.interp(
    df2['timing_from_lap'],
    data_telemetry_pos_lap2026['timing_from_lap'],
    data_telemetry_pos_lap2026['distance_m'],
)

/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_35206/4170714412.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['distance_m'] = np.interp(


In [20]:
_start_2025, _end_2025 = 3345, 4700
df3 = data_telemetry_pos_lap2025[
    (data_telemetry_pos_lap2025['distance_m'] > _start_2025 - 50)
    & (data_telemetry_pos_lap2025['distance_m'] < _end_2025 + 50)
]
df4 = data_telemetry_car_lap2025[
    (data_telemetry_car_lap2025['timing_from_lap'] > df3['timing_from_lap'].min())
    & (data_telemetry_car_lap2025['timing_from_lap'] < df3['timing_from_lap'].max())
]
df4['distance_m'] = np.interp(
    df4['timing_from_lap'],
    data_telemetry_pos_lap2025['timing_from_lap'],
    data_telemetry_pos_lap2025['distance_m'],
)

/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_35206/3746198013.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['distance_m'] = np.interp(


In [32]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=["Speed", "Throttle"],
    # title="Telemetry data on the Shanghai straight",
)
plot_df = df2[(df2['distance_m']>_start_2026) & (df2['distance_m']<_end_2026 + 100)]
plot_df['distance_plt'] = plot_df['distance_m'] - _end_2026
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['speed'].values,
        name='Speed - 2026',
        line=dict(color='blue'),
        marker=dict(size=4),
        mode='lines+markers',
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['throttle'].values,
        name='Throttle - 2026',
        line=dict(color='blue'),
        marker=dict(size=4),
        mode='lines+markers',
    ),
    row=2, col=1,
)
plot_df = df4[(df4['distance_m']>_start_2025) & (df4['distance_m']<_end_2025 + 100)]
plot_df['distance_plt'] = plot_df['distance_m'] - _end_2025
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['speed'].values,
        name='Speed - 2025',
        line=dict(color='green'),
        marker=dict(size=4),
        mode='lines+markers',
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['throttle'].values,
        name='Throttle - 2025',
        line=dict(color='green'),
        marker=dict(size=4),
        mode='lines+markers',
    ),
    row=2, col=1,
)
fig.update_layout(
    title="Telemetry data on the Miami straight (T16 to T17)",
    height=600,
    width=1000,
    hovermode="x unified",
    showlegend=False,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(
    showspikes=True,
    spikemode="across",
    spikedash="solid",
    spikecolor="rgba(128, 128, 128, 0.5)",
    spikethickness=1,
    # title="Distance to apex (m)"
)



/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_35206/207160222.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plot_df['distance_plt'] = plot_df['distance_m'] - _end_2026
/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_35206/207160222.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plot_df['distance_plt'] = plot_df['distance_m'] - _end_2025
